# Storm Night — small EDA

Quick look at Sunday night's data (15 Nov 2026) before writing the classifier.
Run from the repo root (expects `data/`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("data")
LOCAL_TIMEZONE = "Europe/Bratislava"
LOW_CONFIDENCE = 0.7

pd.set_option("display.max_colwidth", 120)


def read_jsonl(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


calls = pd.read_csv(DATA_DIR / "calls.csv", parse_dates=["started_at"]).set_index("call_id")
transcripts = read_jsonl(DATA_DIR / "transcripts.jsonl")
labeled = pd.DataFrame(read_jsonl(DATA_DIR / "labeled_sample.jsonl")).set_index("call_id")
labeled["note"] = labeled["note"].fillna("")

# One row per event, so most questions below are a filter plus a groupby.
events = pd.DataFrame([{"call_id": c["call_id"], **e} for c in transcripts for e in c["events"]])
hangups = events[events["type"] == "hangup"].set_index("call_id")
events["after_hangup"] = events["t"] > events["call_id"].map(hangups["t"])
events["at"] = events["call_id"].map(calls["started_at"]) + pd.to_timedelta(events["t"], unit="s")
# The caller turn that a tool call or a bot reply is reacting to.
events["caller_conf"] = events.groupby("call_id")["asr_conf"].ffill()
events["caller_text"] = events["text"].where(events["type"] == "caller")
events["caller_text"] = events.groupby("call_id")["caller_text"].ffill()
results = events[events["type"] == "tool_result"]

print(len(calls), "rows in calls.csv |", len(transcripts), "transcripts |", len(labeled), "labeled")
print("same call_ids in both files:", set(calls.index) == set(events["call_id"]))

## 1. `calls.csv` — what the dashboard is built from

In [ ]:
ended_by = calls["ended_by"].value_counts()
print(ended_by.to_frame("calls").assign(share=(ended_by / len(calls)).round(3)))
print()
print("flow_version:", calls["flow_version"].value_counts().to_dict())
print("duration_s:", calls["duration_s"].describe()[["min", "25%", "50%", "75%", "max"]].to_dict())

calls_per_caller = calls["caller_hash"].value_counts()
n_repeat_callers = (calls_per_caller > 1).sum()
print(
    f"unique callers: {len(calls_per_caller)}, callers with >1 call: {n_repeat_callers}, "
    f"max calls by one number: {calls_per_caller.max()}"
)

**The 38%:** `ended_by == bot` is 4,332 / 11,400. It only records *who pressed hang up*.
Section 3 shows what is actually behind those calls.

## 2. Transcript structure — event types, statuses (the lists are "not closed")

In [ ]:
fields = {}
for call in transcripts:
    for event in call["events"]:
        fields.setdefault(event["type"], set()).update(event)

print("event types:", events["type"].value_counts().to_dict())
for kind, names in fields.items():
    print(f"  {kind:12s}", sorted(names))
print("\ntool results:")
print(results.groupby(["name", "status"]).size().to_string())
print("\ntransfer:", events.loc[events["type"] == "transfer", "status"].value_counts().to_dict())
print("route:", events.loc[events["type"] == "route", "target"].value_counts().to_dict())
print("hangup by:", hangups["by"].value_counts().to_dict())

In [ ]:
assert hangups.index.is_unique and len(hangups) == len(calls), "every call has one hangup"
assert (hangups["by"] == calls["ended_by"]).all(), "hangup.by matches calls.ended_by"

last_event_types = events.groupby("call_id")["type"].last().value_counts().to_dict()
print("last event type:", last_event_types, "(tool_result last = result arrived AFTER the hangup)")

tool_calls = events[events["type"] == "tool_call"]
unanswered = tool_calls.groupby("name").size() - results.groupby("name").size()
print("tool_call with no result at all:", unanswered[unanswered > 0].to_dict())

## 3. What is behind "the bot hung up"? — last thing the bot said, by `ended_by`

In [ ]:
# (substring of the bot's text, meaning). The first match wins.
BOT_MESSAGE_MEANINGS = [
    ("Ďakujem za zavolanie", "goodbye after outage info"),
    ("Ďakujeme, dovidenia", "goodbye after outage info"),
    ("Hlásenie je založené", "ticket created"),
    ("Prepájam", "transfer to dispatch"),
    ("zákaznícku linku", "routed to customer line"),
    ("adresu sa mi nepodarilo", "FAIL: address not found, gave up"),
    ("dispečeri sú momentálne", "FAIL: dispatch busy"),
    ("technická chyba", "FAIL: tool error apology"),
    ("nemôžem overiť", "FAIL: tool error apology"),
    ("neodpovedá", "FAIL: tool error apology"),
    ("evidujeme poruchu", "outage info (caller hung up)"),
    ("Overím si", "address confirm / waiting for OMS"),
    ("Dobrý večer", "greeting only"),
]


def meaning_of(text: str) -> str:
    for fragment, meaning in BOT_MESSAGE_MEANINGS:
        if fragment in text:
            return meaning
    return f"other: {text[:40]}"


last_bot_text = events[events["type"] == "bot"].groupby("call_id")["text"].last()
calls["last_bot"] = last_bot_text.reindex(calls.index, fill_value="").map(meaning_of)
pd.crosstab(calls["last_bot"], calls["ended_by"], margins=True).sort_values("All", ascending=False)

In [ ]:
bot_hangups = calls[calls["ended_by"] == "bot"]
n_goodbye = bot_hangups["last_bot"].str.startswith("goodbye").sum()
n_after_failure = len(bot_hangups) - n_goodbye
print(
    f"bot hangups: {len(bot_hangups)}  -> normal goodbye: {n_goodbye} "
    f"({n_goodbye / len(bot_hangups):.0%})  | after a failure: {n_after_failure} "
    f"({n_after_failure / len(calls):.1%} of ALL calls)"
)

## 4. Voltika OMS API (`get_outage`) vs our timeouts

In [ ]:
DEGRADED_ERROR_PCT = 10
BUCKET_SIZE = "5min"

oms = pd.read_csv(
    DATA_DIR / "oms_api_metrics.txt",
    sep=r"\s+",
    comment="#",
    names=["bucket", "requests", "p50_ms", "p95_ms", "error_rate"],
    parse_dates=["bucket"],
    index_col="bucket",
)
oms["error_rate"] = oms["error_rate"].str.rstrip("%").astype(float)

outage_results = results[results["name"] == "get_outage"].copy()
outage_results["bucket"] = outage_results["at"].dt.floor(BUCKET_SIZE)
outage_results["is_failed"] = outage_results["status"] != "ok"

fig, ax = plt.subplots(figsize=(9, 3))
oms["error_rate"].plot(ax=ax, label="OMS error rate % (Voltika metrics)")
(outage_results.groupby("bucket")["is_failed"].mean() * 100).plot(
    ax=ax, label="our get_outage non-ok % (transcripts)", alpha=0.7
)
ax.set_title("get_outage failures, 5-min buckets (UTC)")
ax.legend()
plt.tight_layout()
plt.show()

degraded = oms[oms["error_rate"] > DEGRADED_ERROR_PCT].index
print("degraded window (UTC):", degraded.min(), "->", degraded.max() + pd.Timedelta(BUCKET_SIZE))
timeouts = outage_results[outage_results["is_failed"]]
print("timeout latency values:", timeouts["latency_ms"].astype(int).value_counts().to_dict())
n_in_window = timeouts["bucket"].between(degraded.min(), degraded.max()).sum()
print(f"timeouts inside window: {n_in_window} of {len(timeouts)} total")

## 5. Address resolution and ASR confidence (`asr_conf`)

`asr_conf` is the speech-to-text confidence (0–1) on every `caller` turn.

In [ ]:
lookups = results[results["name"] == "resolve_address"].copy()
lookups["attempt"] = lookups.groupby("call_id").cumcount() + 1
lookups["is_failed"] = lookups["status"] != "ok"

attempts = lookups.groupby("call_id").size().reindex(calls.index, fill_value=0)
print("resolve_address attempts per call:", attempts.value_counts().sort_index().to_dict())
print("\nstatus by attempt number (with median asr_conf):")
print(
    pd.crosstab(lookups["attempt"], lookups["status"]).assign(
        median_conf=lookups.groupby("attempt")["caller_conf"].median()
    )
)
conf_bins = pd.cut(lookups["caller_conf"], [0, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
pd.crosstab(conf_bins, lookups["status"])

In [ ]:
# What callers answer to "Overím si: <address>. Je to správne?"
next_event = events.groupby("call_id")[["type", "name", "text"]].shift(-1)
is_confirmation = (events["type"] == "bot") & events["text"].str.startswith("Overím si", na=False)
replies = next_event[is_confirmation]
reply_text = replies["text"].where(replies["type"] == "caller", "<hangup>")

is_rejection = reply_text.str.lower().str.startswith("nie")
rejections = is_rejection.groupby(events.loc[replies.index, "call_id"]).sum()
reply_text.value_counts().to_frame("count")

In [ ]:
caller_turns = events[events["type"] == "caller"].copy()
next_of_caller = next_event.loc[caller_turns.index]
caller_turns["is_address_turn"] = (next_of_caller["type"] == "tool_call") & (
    next_of_caller["name"] == "resolve_address"
)
caller_turns["is_low"] = caller_turns["asr_conf"] < LOW_CONFIDENCE
by_turn = caller_turns.groupby("is_address_turn")
print(
    by_turn["asr_conf"]
    .describe()[["count", "25%", "50%", "75%"]]
    .assign(low_share=by_turn["is_low"].mean().round(3))
)

fig, ax = plt.subplots(figsize=(7, 3))
for is_address_turn, label in [
    (False, "other turns (yes/no, problem, thanks)"),
    (True, "address turns"),
]:
    turns = caller_turns[caller_turns["is_address_turn"] == is_address_turn]
    turns["asr_conf"].plot.hist(
        ax=ax, bins=34, range=(0.3, 1), alpha=0.6, density=True, label=label
    )
ax.axvline(LOW_CONFIDENCE, color="grey", linestyle="--")
ax.set_title("asr_conf of caller turns")
ax.set_xlabel("asr_conf")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
first_attempts = lookups[lookups["attempt"] == 1].set_index("call_id")
first_attempts["is_low"] = first_attempts["caller_conf"] < LOW_CONFIDENCE

# Does asking again help when the first transcript was bad?
address_found = lookups.groupby("call_id")["status"].last() == "ok"
print("first attempt low confidence vs address found in the end:")
print(pd.crosstab(first_attempts["is_low"], address_found, margins=True))

# Storm noise would show up as more low-confidence first attempts later in the night.
local_hour = (
    calls.loc[first_attempts.index, "started_at"]
    .dt.tz_convert(LOCAL_TIMEZONE)
    .dt.hour.rename("local_hour")
)
print("\nfirst address turn by local hour:")
print(first_attempts.groupby(local_hour)["is_low"].agg(turns="size", low_share="mean").round(3))

In [ ]:
# "lomeno" is how ASR writes the slash in house numbers like 5/a.
HOUSE_NUMBER_SLASH = "lomeno"

print("examples of low-confidence address transcripts:")
low = lookups[lookups["caller_conf"] < LOW_CONFIDENCE]
print(low.sample(6, random_state=1)[["caller_conf", "caller_text"]])

confident = lookups[lookups["caller_conf"] >= LOW_CONFIDENCE]
has_slash = confident["caller_text"].str.contains(HOUSE_NUMBER_SLASH).rename("has_slash")
print(f"\nconfident lookups: {len(confident)}, failed: {confident['is_failed'].sum()}")
print("failure rate of confident lookups, by house number with a slash:")
print(confident.groupby(has_slash)["is_failed"].agg(["sum", "count", "mean"]).round(3))

**What `asr_conf` shows**

- Low confidence is an address problem: 13.9% of address turns are below 0.7, compared with 1.5% of
  other turns (short replies like "áno", "hej").
- Below 0.7 the resolver almost never finds the address (1 OK out of ~1,640). The transcripts are
  garbled: words split in two ("štyrid saťosem", "pez inok"), and street names spelled as letters
  ("eš en pe" for SNP).
- Asking again helps: 593 of the 844 calls (70%) whose first address turn was low still found the
  address. Retry turns are worse on average, though (median 0.69 on the 2nd attempt), so some
  callers never get through.
- It did not get worse during the storm: the low share on the first address turn stays at 7–11%
  every hour. So this is our resolver's normal behaviour, not storm noise.
- Confident lookups also fail when the house number has a slash ("lomeno", e.g. 5/a): 33% failed,
  compared with 3.9% without one. That's a resolver parsing bug on our side, separate from ASR.
- For the classifier `asr_conf` isn't needed: the label depends on the outcome, not its cause.

## 6. Other things worth noticing

In [ ]:
# What we put into create_ticket.description
ticket_calls = events[(events["type"] == "tool_call") & (events["name"] == "create_ticket")]
descriptions = ticket_calls["args"].str.get("description").str[:60]
print("create_ticket description values:", descriptions.value_counts().head(3).to_dict())

In [ ]:
# ETA already in the past at the moment the bot read it out
etas = outage_results[outage_results["status"] == "ok"].copy()
etas["eta"] = pd.to_datetime(etas["result"].str.get("eta"), utc=True)
etas = etas.dropna(subset=["eta"])
etas["is_eta_in_past"] = etas["eta"] < etas["at"]
etas["local_hour"] = etas["at"].dt.tz_convert(LOCAL_TIMEZONE).dt.hour
print("ETA already in the past, by local hour:")
print(etas.groupby("local_hour")["is_eta_in_past"].agg(["sum", "count"]))

In [ ]:
# Caller hung up while waiting for get_outage (each call has at most one get_outage)
outage_calls = tool_calls[tool_calls["name"] == "get_outage"].set_index("call_id")
outages = outage_calls[["t"]].join(outage_results.set_index("call_id")[["status", "after_hangup"]])
outages["hangup_by"] = hangups["by"]
outages["hangup_t"] = hangups["t"]

is_waiting = outages["status"].isna() | outages["after_hangup"].eq(True)
waits = outages[is_waiting & (outages["hangup_by"] == "caller")]
waited_s = waits["hangup_t"] - waits["t"]
print(
    "caller hung up while waiting for get_outage:",
    len(waits),
    "| median wait",
    round(waited_s.median(), 1),
    "s",
)
print(waits["status"].fillna("never came").value_counts().to_dict())

## 7. Hand-labeled sample (the reference)

In [ ]:
print(labeled["label"].value_counts().to_dict())
labeled[labeled["note"] != ""].groupby(["label", "note"]).size().to_frame("n")

In [ ]:
# The facts that decide each label, for every call. The sample below shows which combination
# leads to which label.
outage_outcome = outages["status"].fillna("never came")
outage_outcome = outage_outcome.where(
    ~outages["after_hangup"].eq(True), outage_outcome + " after hangup"
)

facts = pd.DataFrame(index=calls.index)
facts["addr_tries"] = attempts
facts["addr_last"] = lookups.groupby("call_id")["status"].last()
facts["rejected"] = rejections.reindex(calls.index, fill_value=0)
facts["outage"] = outage_outcome
facts["ticket"] = results[results["name"] == "create_ticket"].groupby("call_id")["status"].last()
facts["transfer"] = events[events["type"] == "transfer"].groupby("call_id")["status"].last()
facts["routed"] = events[events["type"] == "route"].groupby("call_id")["target"].last()
facts["hangup_by"] = hangups["by"]
facts = facts.fillna("-")

sample = facts.loc[labeled.index].assign(label=labeled["label"])
sample.groupby(["label", *facts.columns]).size().to_frame("n")

In [ ]:
# One example per label, to read the actual dialogue behind each outcome.
def short_dialogue(call_events: list[dict]) -> str:
    steps = []
    for e in call_events:
        if e["type"] == "bot":
            steps.append(f"BOT: {e['text'][:45]}")
        elif e["type"] == "caller":
            steps.append(f"CALLER ({e['asr_conf']}): {e['text'][:45]}")
        elif e["type"] == "tool_result":
            steps.append(f"  -> {e['name']}: {e['status']} at {e['t']}s")
        elif e["type"] == "hangup":
            steps.append(f"HANGUP by {e['by']} at {e['t']}s")
        elif e["type"] in ("transfer", "route"):
            steps.append(f"  -> {e['type']}: {e.get('status', e.get('target'))}")
    return "\n".join(steps)


for call_id, record in labeled.drop_duplicates("label").iterrows():
    print(f"=== {record['label']}  ({call_id})")
    print(short_dialogue(record["events"]))
    print()

**What led to each label in the sample**

- Every label in the 60 calls matches a different combination of the facts above. No combination
  appears under two labels, so rules on structured events can agree with the whole sample.
- The order that resolves them: transfer → route → ticket → `get_outage` outcome → address.
- `KNOWN_OUTAGE_INFORMED`: the address retries and who hung up don't matter (9 bot, 9 caller).
  Callers often hang up right after hearing the ETA. A known outage has `outage_id` and `eta` in
  the result; an unknown one has `outage_id: null`.
- Caller hung up while waiting for `get_outage`: the late result decides. `ok` after the hangup →
  `ENDED_EARLY`; `timeout` after the hangup or no result at all → `API_ERROR`.
- `ADDRESS_NOT_UNDERSTOOD` has two shapes: three failed lookups, or a resolver that returned an
  address twice and the caller said "nie…" both times. The second is the only case where the
  caller's words matter.
- `TRANSFER_FAILED` and `TRANSFERRED` can happen with no address at all: danger calls skip it.
- Not covered by the sample, so it has to come from the label definitions: an unknown outage where
  the caller hung up before the ticket, a `create_ticket` that never returns, and unknown statuses.

## Summary of findings

1. **"38%" is not a failure rate.** It's `ended_by == bot`, i.e. who pressed hang up. About **70% of bot hangups
   (3,028) are a normal "goodbye" after the caller was told the outage and ETA**. Only ~1,300 bot hangups
   (~11% of all calls) follow a failure. Some failures also end with the *caller* hanging up (e.g. waiting on OMS),
   so the real failure count comes from the classifier, not `ended_by`.
2. **Voltika's OMS API was degraded 20:40–21:30 UTC (21:40–22:30 local).** Error rate was ~30% and p95 hit the 8 s
   timeout. Our `get_outage` timeouts (592, all at exactly 8000 ms) sit almost entirely in that window, and the
   bot apologized and hung up. **Owner: Voltika (API) + product (no fallback, e.g. create a ticket anyway).**
3. **Address resolution is the biggest of *our* problems.** 2,071 of 11,642 lookups failed (no_match/ambiguous).
   **Lookups with ASR confidence < 0.7 failed almost every time (1 OK out of ~1,640)**, so it looks like a confidence cut-off. 456 calls ended with
   "address not found", and some callers rejected the resolved address ("nie to nie je moja adresa").
4. **Dispatch transfers:** 798 connected, 228 failed (160 queue full at the 10-caller limit, 68 timeouts).
5. **Bug in our flow:** every `create_ticket.description` is the bot's own prompt text ("Na vašej adrese zatiaľ
   poruchu neevidujeme…"), never what the caller said. All ~2,100 tickets lack the caller's description.
6. **Stale ETAs:** after 23:00 local, ~15–30% of ETAs from OMS were already in the past when the bot read them out.
   That's Voltika data, and **a direct risk for the SMS request**.
7. **337 callers hung up while waiting for `get_outage`** (median ~2.5 s). Some results came back OK later
   (→ ENDED_EARLY), some timed out or never came (→ API_ERROR, per the labeled notes).
8. **815 numbers called more than once** (1,915 calls). Repeats inflate the volume, and most of those first calls
   ended at the greeting or in a failure, so people called back.
9. Data is clean: ids match, every call has exactly one hangup, `ended_by` matches the transcript, single flow `v14`.
10. **ASR confidence explains most address failures, but not all.** 14% of address turns are below
    0.7 (vs 1.5% of other turns), and those almost never resolve. A retry rescues 70% of them. The
    rate is flat all night, so it's our normal behaviour, not the storm. Separately, house numbers
    with a slash ("lomeno") fail 33% of the time even when ASR is confident: a resolver bug (ours).
11. **The labeled sample is fully explained by structured events** (transfer, route, ticket,
    `get_outage` outcome relative to the hangup, address attempts and rejections). A rule-based
    classifier is enough; caller text is only needed to spot "nie, to nie je moja adresa".